In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import cv2
import numpy as np
from PIL import Image, ImageTk
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import time

## Image Operations

In [2]:
def op_brightness(img, val):
    return cv2.convertScaleAbs(img, alpha=1, beta=val)

def op_contrast(img, val):
    return cv2.convertScaleAbs(img, alpha=val, beta=0)

def op_gamma(img, val):
    table = np.array([(i/255.0)**(1.0/val)*255 for i in range(256)], dtype=np.uint8)
    return cv2.LUT(img, table)

def op_gaussian(img, k):
    k = k if k%2==1 else k+1
    return cv2.GaussianBlur(img, (k, k), 0)

def op_median(img, k):
    k = k if k%2==1 else k+1
    return cv2.medianBlur(img, k)

def op_canny(img, t1, t2):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img
    return cv2.cvtColor(cv2.Canny(gray, t1, t2), cv2.COLOR_GRAY2BGR)

def op_sobel(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx**2+sy**2)
    mag = np.clip(mag/mag.max()*255, 0, 255).astype(np.uint8)
    return cv2.cvtColor(mag, cv2.COLOR_GRAY2BGR)

def op_hist_gray(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img
    return cv2.cvtColor(cv2.equalizeHist(gray), cv2.COLOR_GRAY2BGR)

def op_hist_color(img):
    yuv = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)
    yuv[:,:,0] = cv2.equalizeHist(yuv[:,:,0])
    return cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)

def op_nearest(img, scale):
    h, w = img.shape[:2]
    return cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_NEAREST)

def op_bilinear(img, scale):
    h, w = img.shape[:2]
    return cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_LINEAR)

def op_rotate(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
    return cv2.warpAffine(img, M, (w, h))

OPERATIONS = {
    "brightness": op_brightness, "contrast":   op_contrast,
    "gamma":      op_gamma,      "gaussian":   op_gaussian,
    "median":     op_median,     "canny":      op_canny,
    "sobel":      op_sobel,      "hist_gray":  op_hist_gray,
    "hist_color": op_hist_color, "nearest":    op_nearest,
    "bilinear":   op_bilinear,   "rotate":     op_rotate,
}

## Fire Detector

In [3]:
def _build_fire_mask(img):
    if len(img.shape)==2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    m1 = cv2.inRange(hsv, np.array([0,  120,100]), np.array([22, 255,255]))
    m2 = cv2.inRange(hsv, np.array([158,120,100]), np.array([180,255,255]))
    m3 = cv2.inRange(hsv, np.array([15,  30,200]), np.array([40, 180,255]))
    m4 = cv2.inRange(hsv, np.array([0,  150, 50]), np.array([15, 255,120]))
    b, g, r = cv2.split(img)
    rgb_gate = ((r.astype(np.int16)-b.astype(np.int16))>20).astype(np.uint8)*255
    mask = cv2.bitwise_or(cv2.bitwise_or(m1,m2), cv2.bitwise_or(m3,m4))
    return cv2.bitwise_and(mask, rgb_gate)


def detect_fire(img):
    """Returns (annotated_bgr, fire_pixels, fire_pct, region_count, avg_conf, elapsed_ms)"""
    if len(img.shape)==2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    t0   = time.perf_counter()
    mask = _build_fire_mask(img)
    h, w = img.shape[:2]
    k    = max(5, int(min(h,w)*0.007)|1)
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k,k))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kern)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kern)
    elapsed_ms  = (time.perf_counter()-t0)*1000
    fire_pixels = int(np.sum(mask>0))
    total_px    = h*w
    fire_pct    = fire_pixels/total_px*100
    min_area    = max(100, int(total_px*0.0005))

    result   = img.copy()
    contours,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    region_count = 0
    conf_list    = []

    for cnt in contours:
        if cv2.contourArea(cnt)<min_area: continue
        region_count += 1
        x,y,bw,bh = cv2.boundingRect(cnt)
        roi  = mask[y:y+bh, x:x+bw]
        conf = int(np.sum(roi>0)/(bw*bh)*100) if bw*bh>0 else 0
        conf_list.append(conf)
        col = (0,0,255) if conf>=75 else (0,100,255) if conf>=50 else (0,160,255)
        cv2.rectangle(result,(x,y),(x+bw,y+bh),col,2)
        cv2.putText(result,f"FIRE {conf}%",(x,max(y-6,12)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.52,col,1,cv2.LINE_AA)

    if   fire_pct==0: sev,sc="NO FIRE",(0,200,0)
    elif fire_pct< 2: sev,sc="LOW",(0,200,200)
    elif fire_pct<10: sev,sc="MEDIUM",(0,165,255)
    elif fire_pct<25: sev,sc="HIGH",(0,60,255)
    else:             sev,sc="CRITICAL",(0,0,200)

    badge = f"[ {sev} ]  {fire_pct:.1f}%  |  {region_count} region(s)"
    cv2.rectangle(result,(0,0),(len(badge)*9+10,22),(20,20,20),-1)
    cv2.putText(result,badge,(5,15),cv2.FONT_HERSHEY_SIMPLEX,0.50,sc,1,cv2.LINE_AA)

    ov = result.copy(); ov[mask>0]=[0,40,220]
    cv2.addWeighted(ov,0.30,result,0.70,0,result)

    avg_conf = int(np.mean(conf_list)) if conf_list else 0
    return result, fire_pixels, fire_pct, region_count, avg_conf, elapsed_ms


In [ ]:
class VisionEditor:
    def __init__(self, root):
        self.root = root
        self.root.title("Vision Editor – Fire Detection")
        self.root.geometry("1300x920")
        self.root.configure(bg="#0a0a0a")
        self.original_img    = None
        self.working_img     = None
        self.hist_mode       = "Gray"
        self.current_mode    = None
        self.cumulative_mode = tk.BooleanVar(value=False)
        self._build_ui()

   
    def _build_ui(self):
        
        mbar = tk.Frame(self.root, bg="#0a0a0a")
        mbar.pack(fill="x")

        def make_menu(parent, lbl, items):
            btn = tk.Menubutton(parent, text=lbl, bg="#0a0a0a", fg="#e8d5b0",
                                activebackground="#1a1a1a", activeforeground="#e8d5b0",
                                font=("Arial",9,"bold"), padx=10, pady=4,
                                relief="flat", bd=0)
            btn.pack(side="left")
            m = tk.Menu(btn, tearoff=0, bg="#1a1a1a", fg="#e8d5b0",
                        activebackground="#e8d5b0", activeforeground="#111100",
                        font=("Arial",9))
            btn.config(menu=m)
            for item in items:
                if item is None: m.add_separator()
                else: m.add_command(label=item[0], command=item[1])

        make_menu(mbar," File",[("Open Image",self.open_image),None,("Exit",self.root.quit)])
        make_menu(mbar," Tools",[
            ("Brightness",   lambda:self.set_mode("brightness")),
            ("Contrast",     lambda:self.set_mode("contrast")),
            ("Gamma",        lambda:self.set_mode("gamma")),
            ("Gaussian Blur",lambda:self.set_mode("gaussian")),
            ("Median Blur",  lambda:self.set_mode("median")),
            ("Canny",        lambda:self.set_mode("canny")),
            ("Sobel",        lambda:self.set_mode("sobel")),
            ("Nearest",      lambda:self.set_mode("nearest")),
            ("Bilinear",     lambda:self.set_mode("bilinear")),
            ("Rotate",       lambda:self.set_mode("rotate")),
        ])
        make_menu(mbar," Histogram",[
            ("Histogram Gray",  lambda:self.set_mode("hist_gray")),
            ("Histogram Color", lambda:self.set_mode("hist_color")),
            None,
            ("Toggle Gray/RGB", self.toggle_hist_mode),
        ])

        
        top = tk.Frame(self.root, bg="#0a0a0a", pady=5)
        top.pack(fill="x")

        tk.Button(top, text="⟳ Reset", command=self.reset_to_original,
                  bg="#ff4444", fg="white", font=("Arial",9,"bold"),
                  relief="flat", padx=8).pack(side="left", padx=5)

        tk.Checkbutton(top, text="Cumulative (stack filters)",
                       variable=self.cumulative_mode,
                       bg="#0a0a0a", fg="#e8d5b0", selectcolor="#1a1a1a",
                       activebackground="#0a0a0a", activeforeground="#e8d5b0",
                       font=("Arial",9)).pack(side="left", padx=12)

        tk.Button(top, text="🔥  Detect Fire  —  Original vs Processed",
                  command=self.run_fire_compare,
                  bg="#aa2200", fg="white", font=("Arial",9,"bold"),
                  relief="flat", padx=14).pack(side="left", padx=10)

        self.status_var = tk.StringVar(value="No filter selected")
        tk.Label(top, textvariable=self.status_var,
                 bg="#0a0a0a", fg="#888888", font=("Arial",8)).pack(side="left", padx=8)

        
        self.slider_frame = tk.Frame(self.root, bg="#1a1a1a", height=70)
        self.slider_frame.pack(fill="x", padx=10, pady=4)
        self.sliders = {}

        
        img_row = tk.Frame(self.root, bg="#0a0a0a")
        img_row.pack(fill="both", expand=True, padx=4, pady=4)

        left = tk.Frame(img_row, bg="#0a0a0a")
        left.pack(side="left", expand=True, fill="both")
        tk.Label(left, text="Original Image", bg="#0a0a0a", fg="#aaaaaa",
                 font=("Arial",9,"bold")).pack()
        self.canvas_orig = tk.Label(left, text="Open an image",
                                    bg="#111111", fg="#555555")
        self.canvas_orig.pack(expand=True, fill="both", padx=4, pady=4)

        right = tk.Frame(img_row, bg="#0a0a0a")
        right.pack(side="left", expand=True, fill="both")
        self.proc_title_var = tk.StringVar(value="Processed Image")
        tk.Label(right, textvariable=self.proc_title_var,
                 bg="#0a0a0a", fg="#aaaaaa", font=("Arial",9,"bold")).pack()
        self.canvas_res = tk.Label(right, text="Apply a filter",
                                   bg="#111111", fg="#555555")
        self.canvas_res.pack(expand=True, fill="both", padx=4, pady=4)

        
        cmp_f = tk.Frame(self.root, bg="#0a0a0a")
        cmp_f.pack(fill="x", padx=10, pady=(0,4))

        tk.Label(cmp_f, text="🔥  Fire Detection — Comparison Report",
                 bg="#0a0a0a", fg="#ff9933",
                 font=("Arial",10,"bold")).pack(anchor="w", pady=(4,2))

        tbl_wrap = tk.Frame(cmp_f, bg="#1a1a1a")
        tbl_wrap.pack(fill="x")

        cols = ("Metric","Original","Processed","Δ Change","Winner")
        sty = ttk.Style(); sty.theme_use("default")
        sty.configure("Cmp.Treeview",
                       background="#1a1a1a", foreground="#e8d5b0",
                       fieldbackground="#1a1a1a", rowheight=24, font=("Arial",9))
        sty.configure("Cmp.Treeview.Heading",
                       background="#2a1500", foreground="#ff9933",
                       font=("Arial",9,"bold"))
        sty.map("Cmp.Treeview",
                background=[("selected","#3a1800")],
                foreground=[("selected","#ffffff")])

        self.cmp_table = ttk.Treeview(tbl_wrap, columns=cols,
                                       show="headings", height=6,
                                       style="Cmp.Treeview")
        for c,w in zip(cols,[220,170,170,115,265]):
            self.cmp_table.heading(c, text=c)
            self.cmp_table.column(c, width=w, anchor="center")
        self.cmp_table.pack(fill="x", side="left", expand=True)
        sb = ttk.Scrollbar(tbl_wrap, orient="vertical", command=self.cmp_table.yview)
        self.cmp_table.configure(yscroll=sb.set); sb.pack(side="right", fill="y")

        self.verdict_var = tk.StringVar(value="Press   Detect Fire  to compare.")
        tk.Label(cmp_f, textvariable=self.verdict_var,
                 bg="#0a0a0a", fg="#aaffaa", font=("Arial",9,"italic"),
                 wraplength=980, justify="left").pack(anchor="w", pady=(3,0))

        hist_c = tk.Frame(self.root, bg="#0a0a0a", height=120)
        hist_c.pack(fill="x", side="bottom", padx=4)
        self.fig, (self.ax_orig, self.ax_res) = plt.subplots(1,2,figsize=(9,1.6))
        self.fig.patch.set_facecolor("#0a0a0a")
        self.hist_canvas = FigureCanvasTkAgg(self.fig, master=hist_c)
        self.hist_canvas.get_tk_widget().pack(fill="both", expand=True)

        self.slider_configs = {
            "brightness": [("Brightness",-100,100,0,1)],
            "contrast":   [("Contrast",0.1,3.0,1.0,0.1)],
            "gamma":      [("Gamma",0.1,3.0,1.0,0.1)],
            "gaussian":   [("Blur Size",1,31,1,2)],
            "median":     [("Median Size",1,31,1,2)],
            "canny":      [("Th1",0,255,100,1),("Th2",0,255,200,1)],
            "nearest":    [("Scale",0.1,5.0,2.0,0.1)],
            "bilinear":   [("Scale",0.1,5.0,2.0,0.1)],
            "rotate":     [("Angle",-180,180,0,1)],
        }


    def run_fire_compare(self):
        if self.original_img is None:
            messagebox.showwarning("No Image","Open an image first.")
            return
        proc = self.working_img if self.working_img is not None else self.original_img

        o_ann,o_px,o_pct,o_reg,o_conf,o_ms = detect_fire(self.original_img)
        p_ann,p_px,p_pct,p_reg,p_conf,p_ms = detect_fire(proc)

        self._show_cv(o_ann, self.canvas_orig)
        self._show_cv(p_ann, self.canvas_res)
        self._fill_table(o_px,o_pct,o_reg,o_conf,o_ms,
                         p_px,p_pct,p_reg,p_conf,p_ms)

    def _fill_table(self, o_px,o_pct,o_reg,o_conf,o_ms,
                          p_px,p_pct,p_reg,p_conf,p_ms):
        for r in self.cmp_table.get_children(): self.cmp_table.delete(r)

        def sgn(v, fmt=".1f"):
            return f"+{v:{fmt}}" if v>=0 else f"{v:{fmt}}"

        def win(o,p, higher=True):
            if abs(o-p)<0.01: return "—  Tie"
            return (" Processed" if (p>o)==higher else " Original")

        rows = [
            ("Fire Pixels",
             f"{o_px:,} px", f"{p_px:,} px",
             sgn(p_px-o_px,".0f")+" px", win(o_px,p_px)),

            ("Coverage (%)",
             f"{o_pct:.2f}%", f"{p_pct:.2f}%",
             sgn(p_pct-o_pct)+"%", win(o_pct,p_pct)),

            ("Regions Detected",
             str(o_reg), str(p_reg),
             sgn(p_reg-o_reg,".0f"), win(o_reg,p_reg)),

            ("Avg Confidence (%)",
             f"{o_conf}%", f"{p_conf}%",
             sgn(p_conf-o_conf,".0f")+"%", win(o_conf,p_conf)),

            ("Detection Time (ms)",
             f"{o_ms:.2f} ms", f"{p_ms:.2f} ms",
             sgn(p_ms-o_ms)+" ms", win(o_ms,p_ms, higher=False)),
        ]
        for r in rows: self.cmp_table.insert("","end",values=r)

        pw = sum(1 for r in rows if r[4].startswith(" Processed"))
        ow = sum(1 for r in rows if r[4].startswith(" Original"))
        n  = len(rows)

        if o_pct==0 and p_pct==0:
            v = "ℹ  No fire detected in either image."
        elif pw>ow:
            v=(f"  Processed wins — coverage {o_pct:.2f}% → {p_pct:.2f}%  |  "
               f"regions {o_reg} → {p_reg}  |  avg confidence {o_conf}% → {p_conf}%  "
               f"({pw}/{n} metrics improved).")
        elif ow>pw:
            v=(f"  Original wins — the filter reduced fire-detection accuracy  "
               f"({ow}/{n} metrics better on the original).")
        else:
            v=(f"ℹ  Roughly equal  ({pw} metrics better after filter, {ow} before).")
        self.verdict_var.set(v)

    def _show_cv(self, cv_img, widget):
        rgb = (cv2.cvtColor(cv_img,cv2.COLOR_GRAY2RGB)
               if len(cv_img.shape)==2
               else cv2.cvtColor(cv_img,cv2.COLOR_BGR2RGB))
        im = Image.fromarray(rgb); im.thumbnail((520,360))
        tk_im = ImageTk.PhotoImage(im)
        widget.config(image=tk_im, text=""); widget.image=tk_im

    def _update_hists(self):
        if self.original_img is None: return
        proc = self.working_img if self.working_img is not None else self.original_img
        for ax,img,title in zip([self.ax_orig,self.ax_res],
                                 [self.original_img,proc],
                                 ["Original","Processed"]):
            ax.clear(); ax.set_facecolor("#111111")
            ax.set_title(title,color="white",fontsize=7)
            if self.hist_mode=="RGB" and len(img.shape)==3:
                for i,c in enumerate(["b","g","r"]):
                    ax.plot(cv2.calcHist([img],[i],None,[256],[0,256]),color=c,lw=0.8)
            else:
                g = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img
                ax.plot(cv2.calcHist([g],[0],None,[256],[0,256]),color="white",lw=0.8)
            ax.set_xlim([0,256]); ax.tick_params(labelsize=5,colors="gray")
        self.hist_canvas.draw()

    def toggle_hist_mode(self):
        self.hist_mode = "RGB" if self.hist_mode=="Gray" else "Gray"
        self._update_hists()

    def open_image(self):
        path = filedialog.askopenfilename(
            filetypes=[("Image files","*.jpg *.jpeg *.png *.bmp")])
        if not path: return
        img = cv2.imread(path)
        if img is None:
            messagebox.showerror("Error","Could not open image."); return
        self.original_img = img.copy()
        self.working_img  = img.copy()
        self._show_cv(self.original_img, self.canvas_orig)
        self._show_cv(self.working_img,  self.canvas_res)
        self.proc_title_var.set("Processed Image  (no filter yet)")
        self._update_hists()
        self.verdict_var.set("Press  Detect Fire  to compare.")
        for r in self.cmp_table.get_children(): self.cmp_table.delete(r)

    def set_mode(self, mode):
        self.current_mode = mode
        for w in self.slider_frame.winfo_children(): w.destroy()
        self.sliders.clear()
        self.status_var.set(f"Active Filter: {mode}")
        no_slider = ["hist_gray","hist_color","sobel"]
        if mode in no_slider:
            self.apply_filter()
        elif mode in self.slider_configs:
            for lbl,lo,hi,default,res in self.slider_configs[mode]:
                sf = tk.Frame(self.slider_frame, bg="#1a1a1a"); sf.pack(side="left",padx=20)
                tk.Label(sf,text=lbl,bg="#1a1a1a",fg="white",font=("Arial",7)).pack()
                var = tk.DoubleVar(value=default)
                s = tk.Scale(sf,variable=var,from_=lo,to=hi,resolution=res,
                             orient="horizontal",bg="#1a1a1a",fg="#e8d5b0",
                             highlightthickness=0,length=150)
                s.pack()
                s.bind("<ButtonRelease-1>", lambda e: self.apply_filter())
                self.sliders[lbl] = var

    def apply_filter(self):
        if self.original_img is None: return
        try:
            src = (self.working_img.copy() if self.cumulative_mode.get()
                   else self.original_img.copy())
            m = self.current_mode
            if   m=="brightness": img=OPERATIONS["brightness"](src,int(self.sliders["Brightness"].get()))
            elif m=="contrast":   img=OPERATIONS["contrast"](src,self.sliders["Contrast"].get())
            elif m=="gamma":      img=OPERATIONS["gamma"](src,self.sliders["Gamma"].get())
            elif m=="gaussian":   img=OPERATIONS["gaussian"](src,int(self.sliders["Blur Size"].get()))
            elif m=="median":     img=OPERATIONS["median"](src,int(self.sliders["Median Size"].get()))
            elif m=="canny":      img=OPERATIONS["canny"](src,int(self.sliders["Th1"].get()),int(self.sliders["Th2"].get()))
            elif m=="sobel":      img=OPERATIONS["sobel"](src)
            elif m=="hist_gray":  img=OPERATIONS["hist_gray"](src)
            elif m=="hist_color": img=OPERATIONS["hist_color"](src)
            elif m=="nearest":    img=OPERATIONS["nearest"](src,self.sliders["Scale"].get())
            elif m=="bilinear":   img=OPERATIONS["bilinear"](src,self.sliders["Scale"].get())
            elif m=="rotate":     img=OPERATIONS["rotate"](src,self.sliders["Angle"].get())
            else: return
            self.working_img = img
            self.proc_title_var.set(f"Processed  [{m}]")
            self._show_cv(self.working_img, self.canvas_res)
            self._update_hists()
        except Exception as e:
            print(f"Filter Error: {e}")

    def reset_to_original(self):
        if self.original_img is None: return
        self.working_img = self.original_img.copy()
        self._show_cv(self.original_img, self.canvas_orig)
        self._show_cv(self.working_img,  self.canvas_res)
        self.proc_title_var.set("Processed Image  (reset)")
        self._update_hists()
        self.status_var.set("No filter selected")
        self.current_mode = None
        for w in self.slider_frame.winfo_children(): w.destroy()
        self.verdict_var.set("Press   Detect Fire  to compare.")
        for r in self.cmp_table.get_children(): self.cmp_table.delete(r)


if __name__ == "__main__":
    root = tk.Tk()
    app  = VisionEditor(root)
    root.mainloop()